# IKG Column Lineage Master Auto Refresh

Use this notebook to regenerate the column-level lineage Excel report from the GitLab SQL scripts and push the output into `sandbox_prj_smart_insights.ikg_column_lineage_master_auto_refresh`. The code mirrors the standalone Python script and is safe to rerun whenever the SQL sources change.

This notebook extracts comprehensive column-level lineage including:
- Source and target columns for all SELECT statements
- CTE (WITH clause) resolution to trace back to original source tables
- JOIN, WHERE, and HAVING clause column dependencies
- Support for temp tables, subqueries, and complex SQL constructs

In [ ]:
import datetime
import logging

from ikg_column_lineage_master_auto_refresh import (
    EXCLUDE_FOLDER,
    GitLabSQLFetcher,
    ColumnLineageParser,
    ColumnLineageBuilder,
    DatabaseUploader,
    rows_to_dataframe,
    write_to_excel,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

print("Notebook logger initialized.")

In [ ]:
import getpass

PRIVATE_TOKEN = getpass.getpass("Enter your private token: ")
DB_PASSWORD = getpass.getpass("Enter Password for DB User: ")

In [ ]:
exclude_folders = [folder for folder in EXCLUDE_FOLDER.split() if folder]
fetcher = GitLabSQLFetcher(private_token=PRIVATE_TOKEN, exclude_folders=exclude_folders)
parser = ColumnLineageParser()
builder = ColumnLineageBuilder(fetcher, parser)

run_ts = datetime.datetime.utcnow()
output_file = write_to_excel(rows_to_dataframe([]), run_timestamp=run_ts)

print(f"Initialized Excel snapshot at {output_file}")


def flush_excel(current_rows):
    snapshot_df = rows_to_dataframe(current_rows)
    write_to_excel(snapshot_df, output_path=output_file)


rows = builder.build(progress_callback=flush_excel, run_timestamp=run_ts)
df = rows_to_dataframe(rows)
print(f"Collected {len(df)} column lineage rows from GitLab.")
df.head(20)

In [ ]:
print("Excel report is being updated at:")
output_file

In [ ]:
# Optional: View sample of the data
print("Sample column lineage data:")
print(f"Total rows: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nTarget tables processed: {df['sub_target_table'].nunique()}")
print(f"Source tables referenced: {df['source_table'].nunique()}")
print(f"\nSQL process types: {df['sql_process'].value_counts().to_dict()}")

In [ ]:
db_config = {
    "host": "greenplum-rdsp.zur.swissbank.com",
    "port": "5432",
    "dbname": "gprdsp",
    "user": "ds_rdsp_dev",
    "password": DB_PASSWORD,
}

db_uploader = DatabaseUploader(db_config)
db_uploader.refresh_table(rows)
print("Database table refreshed successfully.")
print(f"Table: sandbox_prj_smart_insights.ikg_column_lineage_master_auto_refresh")
print(f"Rows inserted: {len(rows)}")